# Extractor de Datos SIPA - Fase 1
**CU-01: Preprocesar boletin de precios**

Extrae datos de boletines SIPA (PDF con tablas de imagen) usando OCR (tesseract).

In [ ]:
import os
import re
import pdfplumber
import pytesseract
from PIL import Image
import pandas as pd

# Configuracion de tesseract
TESSERACT_PATH = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
LOCAL_TESSDATA = os.path.join(os.getcwd(), "tessdata")

pytesseract.pytesseract.tesseract_cmd = TESSERACT_PATH
if os.path.exists(LOCAL_TESSDATA):
    os.environ["TESSDATA_PREFIX"] = LOCAL_TESSDATA

MESES = {
    "ENERO": 1, "FEBRERO": 2, "MARZO": 3, "ABRIL": 4, "MAYO": 5, "JUNIO": 6,
    "JULIO": 7, "AGOSTO": 8, "SEPTIEMBRE": 9, "OCTUBRE": 10,
    "NOVIEMBRE": 11, "DICIEMBRE": 12
}

PROVINCIAS = ["AZUAY", "GUAYAS", "PICHINCHA"]
OCR_RESOLUTION = 300

# Coordenadas calibradas para boletines 2026 (formato apaisado)
BBOX_PERECEDEROS_FULL = (28, 142, 814, 538)
BBOX_NO_PERECEDEROS_FULL = (27, 42, 814, 296)

print("Configuracion cargada.")

## 1. Funciones de Extraccion

In [ ]:
def parse_header(texto_pagina):
    """Extrae metadata del boletin: numero, mes, ano, quincena."""
    m = re.search(
        r"N\u00ba\s*(\d+)\s+([A-Z\u00c1\u00c9\u00cd\u00d3\u00da]+)\s*-\s*(\d{4})\s+(PRIMERA|SEGUNDA)\s+QUINCENA",
        texto_pagina
    )
    if not m:
        return None
    num, mes, anio, quincena = m.groups()
    if mes not in MESES:
        return None
    return {
        "boletin_num": int(num),
        "mes": MESES[mes],
        "a\u00f1o": int(anio),
        "quincena": 1 if quincena == "PRIMERA" else 2,
        "quincena_id": f"{anio}-{MESES[mes]:02d}-Q{1 if quincena == 'PRIMERA' else 2}",
    }


def ocr_region(pagina_pdfplumber, bbox, lang="spa"):
    """Renderiza una region de la pagina y aplica OCR con tesseract."""
    recorte = pagina_pdfplumber.within_bbox(bbox)
    imagen = recorte.to_image(resolution=OCR_RESOLUTION).original
    texto = pytesseract.image_to_string(imagen, lang=lang)
    return texto


def parsear_tabla_precios(texto_ocr):
    """Convierte texto OCR de una tabla de precios en registros."""
    registros = []
    patron_precio = re.compile(r"^[\d]+[.,]?\d{0,2}$")

    for linea in texto_ocr.split("\n"):
        linea = linea.strip()
        if not linea or len(linea) < 5:
            continue
        if any(p in linea.lower() for p in ["producto", "fuente", "sistema", "usd/presentaci\u00f3n"]):
            continue

        tokens = linea.split()
        if len(tokens) < 3:
            continue

        while tokens and patron_precio.match(tokens[0].rstrip("%")):
            tokens.pop(0)

        if len(tokens) < 3:
            continue

        precios_encontrados = []
        indices_precios = []
        for i in range(len(tokens) - 1, -1, -1):
            t = tokens[i].rstrip("%")
            if patron_precio.match(t) or t == "-":
                precios_encontrados.insert(0, t)
                indices_precios.insert(0, i)
                if len(precios_encontrados) == 2:
                    break

        if len(precios_encontrados) < 2:
            continue

        idx_inicio_nombre = indices_precios[0]
        nombre = " ".join(tokens[:idx_inicio_nombre]).strip(" -\u2022")

        if not nombre:
            continue

        nombre_tokens = nombre.split()
        if len(nombre_tokens) > 1:
            ultimo = nombre_tokens[-1]
            if re.match(r"^\d+\.\d{2}$", ultimo):
                nombre_base = " ".join(nombre_tokens[:-1])
                if len(nombre_base) > 10:
                    nombre = nombre_base
                    precios_encontrados.insert(0, ultimo)
                    indices_precios[0] -= 1

        nombre = re.sub(r"\bIb\b", "lb", nombre)
        nombre = re.sub(r"\blb\b", "lb", nombre)
        nombre = re.sub(r"\b1b\b", "lb", nombre)
        nombre = re.sub(r"\b\|b\b", "lb", nombre)
        nombre = re.sub(r"\bIt\b", "lt", nombre)
        nombre = re.sub(r"\b\|t\b", "lt", nombre)
        nombre = re.sub(r"\[", "(", nombre)
        nombre = re.sub(r"\(", "(", nombre)
        nombre = re.sub(r"\)", ")", nombre)
        nombre = re.sub(r"Invemadero", "Invernadero", nombre)
        nombre = re.sub(r"Tiema", "Tierna", nombre)
        nombre = re.sub(r"Tierma", "Tierna", nombre)
        nombre = re.sub(r"Tiera", "Tierna", nombre)
        nombre = re.sub(r"Fr[e\u00e9]jol", "Frejol", nombre)
        nombre = re.sub(r"aprox[_\s:]+", "aprox. ", nombre)
        nombre = re.sub(r"Se[n\u00f1]o", "Seco", nombre)
        nombre = re.sub(r"Se\x82o", "Seco", nombre)
        nombre = re.sub(r"Se\ufffd[o\u00f3]", "Seco", nombre)
        nombre = re.sub(r"\]$", ")", nombre)
        nombre = re.sub(r"Se[^c\d\s]{1,3}o(?=\s*\()", "Seco", nombre)

        nombre = re.sub(r"de 11\b", "de 1 l", nombre)
        nombre = re.sub(r"de 1 1\b", "de 1 l", nombre)
        nombre = re.sub(r"de 1 I\b", "de 1 l", nombre)
        nombre = re.sub(r"de 1I\)", "de 1 l)", nombre)
        nombre = re.sub(r"de 1l\)", "de 1 l)", nombre)
        nombre = re.sub(r"de 1 /\)", "de 1 l)", nombre)
        nombre = re.sub(r"de 1 \|\)", "de 1 l)", nombre)
        nombre = re.sub(r"de 1 \|", "de 1 l", nombre)

        nombre = re.sub(r"\s+[\d]+[,.][\d,.]+$", "", nombre)
        nombre = re.sub(r"\s+\d+\.\d{2,3}$", "", nombre)

        nombre = re.sub(r"\s+\d+%$", "", nombre)
        nombre = re.sub(r"\s+\d{2,4}$", "", nombre)
        nombre = re.sub(r"\s+\d+\.\d+$", "", nombre)
        nombre = re.sub(r"aprox\.\)", "aprox.)", nombre)
        nombre = re.sub(r"\(aprox\.\)", "", nombre)

        while nombre.startswith("(") and nombre.count("(") > nombre.count(")"):
            nombre = nombre[1:]
        nombre = re.sub(r"\)\)$", ")", nombre)

        if "(" in nombre and nombre.count("(") > nombre.count(")"):
            nombre = nombre + ")"

        nombre = re.sub(r"\s+\(\)$", "", nombre)
        nombre = re.sub(r"\(\)\s*\)", ")", nombre)
        nombre = re.sub(r"\(Envase de\)$", "", nombre)
        nombre = nombre.strip()

        def to_float(v):
            if v == "-":
                return None
            v = v.replace(",", ".")
            if "." not in v and len(v) > 4 and v.isdigit():
                v = v[:-2] + "." + v[-2:]
            return float(v)

        registros.append({
            "producto_raw": nombre,
            "precio_anterior": to_float(precios_encontrados[0]),
            "precio_actual": to_float(precios_encontrados[1]),
        })

    return registros


def dividir_en_provincias(bbox_full, n_provincias=3):
    x0, top, x1, bottom = bbox_full
    ancho_col = (x1 - x0) / n_provincias
    return [(x0 + i * ancho_col, top, x0 + (i + 1) * ancho_col, bottom) for i in range(n_provincias)]


def procesar_boletin(pdf_path):
    registros_totales = []
    encabezados_fallidos = []
    orden_global = 0

    with pdfplumber.open(pdf_path) as pdf:
        n_paginas = len(pdf.pages)

        for i in range(0, n_paginas, 2):
            pagina_portada = pdf.pages[i]
            texto_portada = pagina_portada.extract_text() or ""
            info = parse_header(texto_portada)

            if info is None:
                encabezados_fallidos.append(i + 1)
                print(f"  [!] No se pudo leer encabezado en pagina {i+1}, se omite.")
                continue

            print(f"  Procesando boletin No{info['boletin_num']} - {info['quincena_id']} ...")

            bboxes_perec = dividir_en_provincias(BBOX_PERECEDEROS_FULL)
            for provincia, bbox in zip(PROVINCIAS, bboxes_perec):
                texto = ocr_region(pagina_portada, bbox)
                filas = parsear_tabla_precios(texto)
                for f in filas:
                    f.update(info)
                    f["provincia"] = provincia
                    f["categoria"] = "perecedero"
                    f["orden"] = orden_global
                    orden_global += 1
                registros_totales.extend(filas)

            if i + 1 < n_paginas:
                pagina_2 = pdf.pages[i + 1]
                bboxes_noperec = dividir_en_provincias(BBOX_NO_PERECEDEROS_FULL)
                for provincia, bbox in zip(PROVINCIAS, bboxes_noperec):
                    texto = ocr_region(pagina_2, bbox)
                    filas = parsear_tabla_precios(texto)
                    for f in filas:
                        f.update(info)
                        f["provincia"] = provincia
                        f["categoria"] = "no_perecedero"
                        f["orden"] = orden_global
                        orden_global += 1
                    registros_totales.extend(filas)

    return registros_totales, encabezados_fallidos


def validar_calidad(df, encabezados_fallidos):
    problemas = []

    mask_precio_nulo = df["precio_anterior"].isna() | df["precio_actual"].isna()
    for _, row in df[mask_precio_nulo].iterrows():
        problemas.append({
            "tipo": "precio_nulo", "producto": row["producto_raw"],
            "provincia": row["provincia"], "quincena": row["quincena_id"],
            "detalle": "Precio anterior o actual es nulo"
        })

    for _, row in df.iterrows():
        nombre = row["producto_raw"]
        if pd.isna(nombre):
            continue
        if "\ufffd" in nombre or "?" in nombre:
            problemas.append({
                "tipo": "encoding_error", "producto": nombre,
                "provincia": row["provincia"], "quincena": row["quincena_id"],
                "detalle": "Caracteres de encoding incorrecto"
            })
        if re.search(r"\d+\.\d{2,}", nombre):
            problemas.append({
                "tipo": "precio_en_nombre", "producto": nombre,
                "provincia": row["provincia"], "quincena": row["quincena_id"],
                "detalle": "Nombre contiene numero que parece precio"
            })

    total = len(df)
    unicos = set((p["producto"], p["provincia"], p["quincena"]) for p in problemas)
    completos = total - len(unicos)

    return {
        "total_registros": total,
        "registros_completos": completos,
        "registros_con_problema": len(unicos),
        "porcentaje_completitud": round(completos / total * 100, 1) if total > 0 else 0,
        "productos_unicos": df["producto_raw"].nunique(),
        "quincenas": df.groupby(["a\u00f1o", "mes", "quincena"]).ngroups,
        "problemas": problemas,
    }

print("Funciones cargadas.")

## 2. Seleccionar PDF y Procesar

In [ ]:
PDF_PATH = "precios_mayoristas_2026.pdf"

print(f"Procesando: {PDF_PATH}")
registros, encabezados_fallidos = procesar_boletin(PDF_PATH)
print(f"\nTotal registros extraidos: {len(registros)}")

## 3. Resultados

In [ ]:
df = pd.DataFrame(registros)

variacion = (df["precio_actual"] - df["precio_anterior"]) / df["precio_anterior"] * 100
variacion = variacion.replace([float("inf"), float("-inf")], 0)
df["variacion_pct"] = variacion.fillna(0).round(0).astype(int)

if "orden" in df.columns:
    df = df.sort_values("orden").drop(columns=["orden"])

reporte = validar_calidad(df, encabezados_fallidos)

print("=" * 50)
print("REPORTE DE CALIDAD")
print("=" * 50)
print(f"Registros totales:     {reporte['total_registros']}")
print(f"Registros completos:   {reporte['registros_completos']} ({reporte['porcentaje_completitud']}%)")
print(f"Con problemas:         {reporte['registros_con_problema']}")
print(f"Productos unicos:      {reporte['productos_unicos']}")
print(f"Quincenas:             {reporte['quincenas']}")

tipos = {}
for p in reporte["problemas"]:
    tipos.setdefault(p["tipo"], []).append(p)

if tipos:
    print(f"\n--- PROBLEMAS ({len(reporte['problemas'])} total) ---")
    for tipo, items in tipos.items():
        print(f"\n  [{tipo.upper()}] ({len(items)} registros)")
        for e in items[:3]:
            print(f"    - {e['producto']} ({e['provincia']}, {e['quincena']})")
        if len(items) > 3:
            print(f"    ... y {len(items) - 3} mas")
print("=" * 50)

In [ ]:
df.head(20)

In [ ]:
CSV_PATH = "data/processed/dataset_crudo_sipa.csv"

os.makedirs(os.path.dirname(CSV_PATH), exist_ok=True)
df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
print(f"Guardado: {CSV_PATH}")